In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
import os
import pickle
import matplotlib.pyplot as plt

os.chdir('../')
print("Current working directory:", os.getcwd())

In [ ]:
# 1. Load the Data
df = pd.read_csv('telemetry_data.csv')
display(df.head())

In [ ]:
# 2. Encode categorical state feature
encoder = LabelEncoder()
df['sfc_type_encoded'] = encoder.fit_transform(df['sfc_type'])

# Save the encoder mapping for the Simulator to use later
os.makedirs('models', exist_ok=True)
with open('models/label_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

print("Saved -> models/label_encoder.pkl")

In [ ]:
# 3. Define Features
state_features = [
    'sfc_type_encoded', 'network_active_nodes', 'network_free_cpu', 'network_free_bw', 
    'net_max_node_utilization', 'net_bottleneck_bw'
]
action_features = ['action_max_path_distance', 'action_num_unique_nodes']

# Split features and target for Classifier (Model A) - Use only state features
X_cls = df[state_features]
y_cls = df['outcome_accepted']

# Filter Data for Regressor (Model B) - Only accepted requests! Use both state and action features
df_accepted = df[df['outcome_accepted'] == 1].copy()
X_reg = df_accepted[state_features + action_features]
y_reg = df_accepted['outcome_latency']

print(f"Total samples for Classifier: {len(X_cls)}")
print(f"Accepted samples for Regressor: {len(X_reg)}")

In [ ]:
# 4. Train Model A: Admission Control (XGBClassifier)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

classifier = xgb.XGBClassifier(
    objective='binary:logistic', 
    n_estimators=150, 
    max_depth=5, 
    learning_rate=0.1
)
classifier.fit(X_train_c, y_train_c)

# Evaluate
preds_c = classifier.predict(X_test_c)
print(f"Classifier Accuracy: {accuracy_score(y_test_c, preds_c):.4f}")
print(f"Classifier F1-Score: {f1_score(y_test_c, preds_c):.4f}")

In [ ]:
# 5. Train Model B: Latency Optimizer (XGBRegressor)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

regressor = xgb.XGBRegressor(
    objective='reg:squarederror', 
    n_estimators=150, 
    max_depth=5, 
    learning_rate=0.1
)
regressor.fit(X_train_r, y_train_r)

# Evaluate
preds_r = regressor.predict(X_test_r)
rmse = np.sqrt(mean_squared_error(y_test_r, preds_r))
print(f"Regressor Test RMSE: {rmse:.2f}")

In [ ]:
# 6. Save the Ensemble
classifier.save_model('models/kdn_classifier.json')
print("Saved Gatekeeper -> models/kdn_classifier.json")

regressor.save_model('models/kdn_regressor.json')
print("Saved Optimizer -> models/kdn_regressor.json")